In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import json
import os

print("Libraries loaded ✓")

Libraries loaded ✓


In [3]:

# --- Generate synthetic training data ---
# Simulates 200 users completing challenges with realistic SUDS patterns

np.random.seed(42)

NUM_USERS = 200
DOMAINS = ["social", "professional", "romantic"]
LEVELS = [1, 2, 3, 4, 5]

# SUDS bands per level (realistic ranges)
SUDS_BANDS = {
    1: (10, 30),   # Minimal anxiety
    2: (25, 45),   # Mild anxiety
    3: (40, 65),   # Moderate anxiety
    4: (55, 80),   # High anxiety
    5: (75, 100),  # Severe anxiety
}

# 9 challenges per domain, 3 per level (simplified)
challenges = []
for domain in DOMAINS:
    for level in LEVELS:
        for i in range(3):
            challenges.append({
                "challenge_id": f"{domain}_L{level}_{i+1}",
                "domain": domain,
                "level": level,
            })

challenges_df = pd.DataFrame(challenges)
print(f"Total challenges: {len(challenges_df)}")
challenges_df.head(10)

Total challenges: 45


,challenge_id,domain,level
0,social_L1_1,social,1
1,social_L1_2,social,1
2,social_L1_3,social,1
3,social_L2_1,social,2
4,social_L2_2,social,2
5,social_L2_3,social,2
6,social_L3_1,social,3
7,social_L3_2,social,3
8,social_L3_3,social,3
9,social_L4_1,social,4


In [4]:
# --- Simulate user completion histories ---

records = []

for user_id in range(NUM_USERS):
    # Each user has a "comfort zone" — some domains are easier
    domain_comfort = {d: np.random.uniform(0.3, 0.9) for d in DOMAINS}
    
    # Users generally progress through levels in order
    # but some skip ahead, some get stuck
    max_level_reached = {d: np.random.choice([1,2,3,4,5], p=[0.1, 0.2, 0.3, 0.25, 0.15]) for d in DOMAINS}
    
    for domain in DOMAINS:
        comfort = domain_comfort[domain]
        max_level = max_level_reached[domain]
        
        for level in range(1, max_level + 1):
            # Not every challenge at each level gets completed
            domain_challenges = challenges_df[
                (challenges_df["domain"] == domain) & 
                (challenges_df["level"] == level)
            ]
            
            num_completed = np.random.randint(1, len(domain_challenges) + 1)
            selected = domain_challenges.sample(num_completed)
            
            for _, challenge in selected.iterrows():
                # SUDS before — based on level band
                suds_min, suds_max = SUDS_BANDS[level]
                anxiety_before = np.clip(
                    np.random.normal(
                        loc=(suds_min + suds_max) / 2,
                        scale=10
                    ),
                    0, 100
                ).astype(int)
                
                # SUDS after — generally lower, affected by comfort
                reduction = np.random.normal(loc=15 * comfort, scale=8)
                anxiety_after = np.clip(anxiety_before - reduction, 0, 100).astype(int)
                
                # Did the user engage well? (affects future recommendations)
                engaged = np.random.random() < (0.5 + comfort * 0.4)
                
                records.append({
                    "user_id": user_id,
                    "challenge_id": challenge["challenge_id"],
                    "domain": domain,
                    "level": level,
                    "anxiety_before": anxiety_before,
                    "anxiety_after": anxiety_after,
                    "reduction": anxiety_before - anxiety_after,
                    "engaged": engaged,
                })

completions_df = pd.DataFrame(records)
print(f"Total completion records: {len(completions_df)}")
completions_df.head()

Total completion records: 3712


,user_id,challenge_id,domain,level,anxiety_before,anxiety_after,reduction,engaged
0,0,social_L1_1,social,1,11,6,5,True
1,0,social_L1_2,social,1,20,13,7,False
2,0,social_L1_3,social,1,34,13,21,True
3,0,social_L2_2,social,2,24,13,11,True
4,0,social_L3_1,social,3,46,33,13,True


In [5]:
# --- Feature engineering ---
# For each user, compute features that predict which challenge to recommend next

def compute_user_features(user_completions):
    """Given a user's completion history, compute features for recommendation."""
    features = {}
    
    # Overall stats
    features["total_completed"] = len(user_completions)
    features["avg_reduction"] = user_completions["reduction"].mean()
    features["avg_anxiety_before"] = user_completions["anxiety_before"].mean()
    features["avg_anxiety_after"] = user_completions["anxiety_after"].mean()
    features["engagement_rate"] = user_completions["engaged"].mean()
    
    # Per-domain stats
    for domain in DOMAINS:
        domain_data = user_completions[user_completions["domain"] == domain]
        features[f"{domain}_completed"] = len(domain_data)
        features[f"{domain}_max_level"] = domain_data["level"].max() if len(domain_data) > 0 else 0
        features[f"{domain}_avg_reduction"] = domain_data["reduction"].mean() if len(domain_data) > 0 else 0
        features[f"{domain}_avg_anxiety"] = domain_data["anxiety_before"].mean() if len(domain_data) > 0 else 50
    
    # Recent momentum — last 5 completions
    recent = user_completions.tail(5)
    features["recent_avg_reduction"] = recent["reduction"].mean()
    features["recent_engagement"] = recent["engaged"].mean()
    features["recent_max_level"] = recent["level"].max()
    
    # Avoidance signal — which domain has least completions?
    domain_counts = {d: features[f"{d}_completed"] for d in DOMAINS}
    features["most_avoided_domain"] = min(domain_counts, key=domain_counts.get)
    
    return features

# Test on one user
sample = completions_df[completions_df["user_id"] == 0]
print(compute_user_features(sample))

{'total_completed': 15, 'avg_reduction': np.float64(13.266666666666667), 'avg_anxiety_before': np.float64(30.0), 'avg_anxiety_after': np.float64(16.733333333333334), 'engagement_rate': np.float64(0.8), 'social_completed': 7, 'social_max_level': np.int64(3), 'social_avg_reduction': np.float64(11.714285714285714), 'social_avg_anxiety': np.float64(32.57142857142857), 'professional_completed': 6, 'professional_max_level': np.int64(2), 'professional_avg_reduction': np.float64(13.833333333333334), 'professional_avg_anxiety': np.float64(27.833333333333332), 'romantic_completed': 2, 'romantic_max_level': np.int64(2), 'romantic_avg_reduction': np.float64(17.0), 'romantic_avg_anxiety': np.float64(27.5), 'recent_avg_reduction': np.float64(15.2), 'recent_engagement': np.float64(0.8), 'recent_max_level': np.int64(2), 'most_avoided_domain': 'romantic'}


In [6]:
# --- Build training dataset ---
# Target: what level and domain should we recommend next?
# Logic: recommend the next uncompleted level in the domain where
# the user shows the best reduction (ready to progress) OR
# the most avoided domain (needs gentle push)

training_data = []

for user_id in range(NUM_USERS):
    user_data = completions_df[completions_df["user_id"] == user_id]
    if len(user_data) < 3:
        continue
    
    features = compute_user_features(user_data)
    
    # Determine recommended next challenge
    # Strategy: balance between progression and avoidance-breaking
    
    best_domain = None
    best_score = -1
    
    for domain in DOMAINS:
        max_level = features[f"{domain}_max_level"]
        avg_reduction = features[f"{domain}_avg_reduction"]
        completed = features[f"{domain}_completed"]
        
        if max_level >= 5:
            continue  # Domain complete
        
        # Score: high reduction = ready to progress
        # Low completion = needs attention (avoidance signal)
        readiness = avg_reduction / 20  # Normalise
        avoidance_signal = 1 - (completed / max(features["total_completed"], 1))
        
        score = 0.6 * readiness + 0.4 * avoidance_signal
        
        if score > best_score:
            best_score = score
            best_domain = domain
    
    if best_domain is None:
        continue
    
    recommended_level = min(features[f"{best_domain}_max_level"] + 1, 5)
    
    # Encode domain as numeric
    domain_map = {"social": 0, "professional": 1, "romantic": 2}
    
    row = {**features}
    row["most_avoided_domain"] = domain_map.get(row["most_avoided_domain"], 0)
    row["target_domain"] = domain_map[best_domain]
    row["target_level"] = recommended_level
    
    training_data.append(row)

train_df = pd.DataFrame(training_data)
print(f"Training samples: {len(train_df)}")
train_df.head()

Training samples: 200


,total_completed,avg_reduction,avg_anxiety_before,avg_anxiety_after,engagement_rate,social_completed,social_max_level,social_avg_reduction,social_avg_anxiety,professional_completed,...,romantic_completed,romantic_max_level,romantic_avg_reduction,romantic_avg_anxiety,recent_avg_reduction,recent_engagement,recent_max_level,most_avoided_domain,target_domain,target_level
0,15,13.266667,30.000,16.733333,0.800000,7,3,11.714286,32.571429,6,...,2,2,17.000000,27.500000,15.2,0.8,2,2,2,3
1,24,5.250000,42.125,36.875000,0.666667,5,2,12.000000,29.400000,9,...,10,4,8.000000,46.200000,8.4,1.0,4,0,0,3
2,23,5.913043,39.000,33.086957,0.652174,6,3,8.666667,34.666667,5,...,12,5,3.666667,49.083333,2.6,1.0,5,1,0,4
3,16,8.312500,35.875,27.562500,0.812500,7,3,8.285714,32.428571,2,...,7,5,7.000000,45.857143,8.6,0.8,5,1,1,2
4,18,12.444444,39.500,27.055556,0.611111,5,3,5.000000,21.000000,8,...,5,5,13.800000,56.200000,13.8,0.6,5,0,1,4


In [9]:
# --- Train the recommendation model ---

feature_cols = [
    "total_completed", "avg_reduction", "avg_anxiety_before", 
    "avg_anxiety_after", "engagement_rate",
    "social_completed", "social_max_level", "social_avg_reduction", "social_avg_anxiety",
    "professional_completed", "professional_max_level", "professional_avg_reduction", "professional_avg_anxiety",
    "romantic_completed", "romantic_max_level", "romantic_avg_reduction", "romantic_avg_anxiety",
    "recent_avg_reduction", "recent_engagement", "recent_max_level",
    "most_avoided_domain",
]

X = train_df[feature_cols]
y_domain = train_df["target_domain"]
y_level = train_df["target_level"]

# Split
X_train, X_test, yd_train, yd_test, yl_train, yl_test = train_test_split(
    X, y_domain, y_level, test_size=0.2, random_state=42
)

# --- Domain recommender (which domain to focus on) ---
domain_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric="mlogloss",
)
domain_model.fit(X_train, yd_train)
domain_preds = domain_model.predict(X_test)

print("=== DOMAIN RECOMMENDATION ===")
print(f"Accuracy: {accuracy_score(yd_test, domain_preds):.3f}")
print(classification_report(yd_test, domain_preds, target_names=DOMAINS))

# --- Level recommender (what difficulty level) ---
# XGBoost needs 0-indexed classes
yl_train_adj = yl_train - yl_train.min()
yl_test_adj = yl_test - yl_test.min()

level_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric="mlogloss",
)
level_model.fit(X_train, yl_train_adj)
level_preds = level_model.predict(X_test)

# Store the offset so we can convert back later
level_offset = int(yl_train.min())
print(f"Level offset: {level_offset} (add back to predictions)")

print("\n=== LEVEL RECOMMENDATION ===")
print(f"Accuracy: {accuracy_score(yl_test_adj, level_preds):.3f}")
level_names = [f"L{i+level_offset}" for i in sorted(yl_test_adj.unique())]
print(classification_report(yl_test_adj, level_preds, target_names=level_names))

=== DOMAIN RECOMMENDATION ===
Accuracy: 0.750
              precision    recall  f1-score   support

      social       0.64      0.70      0.67        10
professional       0.69      0.85      0.76        13
    romantic       0.92      0.71      0.80        17

    accuracy                           0.75        40
   macro avg       0.75      0.75      0.74        40
weighted avg       0.77      0.75      0.75        40

Level offset: 2 (add back to predictions)

=== LEVEL RECOMMENDATION ===
Accuracy: 0.775
              precision    recall  f1-score   support

          L2       0.88      0.78      0.82         9
          L3       0.83      0.62      0.71         8
          L4       0.71      0.86      0.77        14
          L5       0.78      0.78      0.78         9

    accuracy                           0.78        40
   macro avg       0.80      0.76      0.77        40
weighted avg       0.79      0.78      0.77        40



In [11]:
# --- Feature importance ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Domain model importance
domain_importance = pd.Series(
    domain_model.feature_importances_, index=feature_cols
).sort_values(ascending=True)
domain_importance.plot(kind="barh", ax=axes[0], color="#5ab4b4")
axes[0].set_title("Domain Recommendation — Feature Importance")

# Level model importance
level_importance = pd.Series(
    level_model.feature_importances_, index=feature_cols
).sort_values(ascending=True)
level_importance.plot(kind="barh", ax=axes[1], color="#c4956a")
axes[1].set_title("Level Recommendation — Feature Importance")

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved feature_importance.png")

Saved feature_importance.png


/var/folders/2w/lwbp53zj3cxcmyy_pgkr5vym0000gp/T/ipykernel_25743/3565629453.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# --- Save models ---
import joblib

os.makedirs("../backend/app/ml", exist_ok=True)

joblib.dump(domain_model, "../backend/app/ml/domain_model.pkl")
joblib.dump(level_model, "../backend/app/ml/level_model.pkl")
joblib.dump(feature_cols, "../backend/app/ml/feature_cols.pkl")

print("Models saved to backend/app/ml/")
print(f"Domain model: {domain_model.n_features_in_} features")
print(f"Level model: {level_model.n_features_in_} features")

Models saved to backend/app/ml/
Domain model: 21 features
Level model: 21 features
